In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import optim
from torch import nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
from tqdm.auto import tqdm


from sklearn import datasets
from sklearn.model_selection import train_test_split


# !pip install torchvision
import torchvision
# !pip install torchmetrics
import torchmetrics

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
digits = datasets.load_digits()
X = digits.images   # images
y = digits.target   # labels

# Hold out 20% for final testing.
x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

# Take 25% of the remaining 80% for validation: about 60/20/20 overall.
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train,
)

batch_size = 32

# Images: floating-point tensors shaped (N, 1, 8, 8), scaled to [0, 1].
X_train = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1) / 16.0
X_val = torch.tensor(x_val, dtype=torch.float32).unsqueeze(1) / 16.0
X_test = torch.tensor(x_test, dtype=torch.float32).unsqueeze(1) / 16.0

# Labels: integer class IDs (0-9), with no scaling or channel dimension.
y_train = torch.as_tensor(y_train, dtype=torch.long)
y_val = torch.as_tensor(y_val, dtype=torch.long)
y_test = torch.as_tensor(y_test, dtype=torch.long)

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True
)
val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False
)
test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False
)

print(f"Training: {len(train_dataset)} | Validation: {len(val_dataset)} | Test: {len(test_dataset)}")


Training: 1077 | Validation: 360 | Test: 360


In [ ]:
#defining the layers and forward pass
in_channels = 1
out_channels = 10

class CNN(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(CNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=1) 

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=16, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(16 * 6 * 6, out_channels)


    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)

        x = F.relu(self.conv2(x))
        x = self.pool(x)

        x = x.reshape(x.shape[0], -1) 
        x = self.fc1(x)
        return x

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model = CNN(in_channels, out_channels).to(device)
print(model)

cpu
CNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=1, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=576, out_features=10, bias=True)
)


In [4]:
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    total_images = 0

    for data, targets in train_loader:
        data = data.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        scores = model(data)
        loss = criterion(scores, targets)
        loss.backward()
        optimizer.step()

        batch_size = data.size(0)
        total_loss += loss.item() * batch_size
        total_images += batch_size

    average_loss = total_loss / total_images
    print( f"Epoch: {epoch+1} | Train loss: {average_loss:.6f}")

Epoch: 1 | Train loss: 2.071512
Epoch: 2 | Train loss: 0.934521
Epoch: 3 | Train loss: 0.386828
Epoch: 4 | Train loss: 0.279145
Epoch: 5 | Train loss: 0.204136
Epoch: 6 | Train loss: 0.150074
Epoch: 7 | Train loss: 0.123799
Epoch: 8 | Train loss: 0.103952
Epoch: 9 | Train loss: 0.080834
Epoch: 10 | Train loss: 0.086009
Epoch: 11 | Train loss: 0.070917
Epoch: 12 | Train loss: 0.057377
Epoch: 13 | Train loss: 0.054029
Epoch: 14 | Train loss: 0.044681
Epoch: 15 | Train loss: 0.042122
Epoch: 16 | Train loss: 0.040213
Epoch: 17 | Train loss: 0.036659
Epoch: 18 | Train loss: 0.027636
Epoch: 19 | Train loss: 0.028132
Epoch: 20 | Train loss: 0.023739
Epoch: 21 | Train loss: 0.018945
Epoch: 22 | Train loss: 0.019715
Epoch: 23 | Train loss: 0.017430
Epoch: 24 | Train loss: 0.017067
Epoch: 25 | Train loss: 0.025211
Epoch: 26 | Train loss: 0.016235
Epoch: 27 | Train loss: 0.012393
Epoch: 28 | Train loss: 0.010800
Epoch: 29 | Train loss: 0.008240
Epoch: 30 | Train loss: 0.008264
Epoch: 31 | Train l

In [6]:
model.eval()

correct = 0
total = 0
total_loss = 0.0

with torch.no_grad(): 
    for data, targets in val_loader:
        data = data.to(device)
        targets = targets.to(device)

        scores = model(data)         
        loss = criterion(scores, targets)

        predictions = scores.argmax(dim=1) 

        correct += (predictions == targets).sum().item()
        total += targets.size(0)
        total_loss += loss.item() * targets.size(0)

accuracy = 100 * correct / total
average_loss = total_loss / total

print(f"Validation accuracy: {accuracy:.3f}%")
print(f"Validation loss:     {average_loss:.7f}")

Validation accuracy: 98.611%
Validation loss:     0.0358263


In [15]:
from torchvision import datasets, transforms
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from torch import optim
from torch import nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm
from tqdm.auto import tqdm
from torch.utils.data import random_split

# !pip install torchvision
import torchvision
# !pip install torchmetrics
import torchmetrics

In [ ]:
#MNIST dataset 
batch_size = 32


train_dataset = datasets.MNIST(
    root="docs/",
    download=True,
    train=True,
    transform=transforms.ToTensor()
)

mean = train_dataset.data.float().mean() / 255
std = train_dataset.data.float().std() / 255

transform = transforms.Compose([
    transforms.ToTensor(),                          
    transforms.Normalize((mean,), (std,))      
])

train_dataset.transform = transform #normalize and tensor before split

train_dataset, val_dataset = random_split(
    train_dataset,
    [0.8, 0.2],
    generator=torch.Generator().manual_seed(42)
)



test_dataset = datasets.MNIST(root="docs/", download=True, train=False, transform=transform)



test_loader = DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

#only shuffle train dataset
train_loader = DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)



print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Test:", len(test_dataset))


Training: 48000
Validation: 12000
Test: 10000


In [32]:
#This model we can utilize pooliung with bigger stride size since 28x28 isntead of 8x8
#And probably need more layers
#same chanenls as before
in_channels = 1
out_channels = 10
class CNN2(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(CNN2, self).__init__()

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) 

        self.conv1 = nn.Conv2d(in_channels=in_channels, out_channels=32, kernel_size=3, padding=1)

        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)

        self.conv3 = nn.Conv2d(in_channels=32, out_channels=16, kernel_size=3, padding=1)

        #28/4=14
        #14/2=7
        self.fc1 = nn.Linear(16 * 3 * 3, out_channels)


    def forward(self, x):
        #print("Input:", x.shape)
        x = F.relu(self.conv1(x))
        x = self.pool(x)

        x = F.relu(self.conv2(x))
        x = self.pool(x)
        #print("After pool1:", x.shape)

        x = F.relu(self.conv3(x))
        x = self.pool(x)
        #print("After pool2:", x.shape)

        x = x.reshape(x.shape[0], -1) 
        #print("After flatten:", x.shape)

        x = self.fc1(x)
        #print("After fc1:", x.shape)

        return x

images, labels = next(iter(train_loader))

print("Images:", images.shape)
print("Labels:", labels.shape)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model_2 = CNN2(in_channels, out_channels).to(device)
print(model_2)

Images: torch.Size([40, 1, 28, 28])
Labels: torch.Size([40])
cpu
CNN2(
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=144, out_features=10, bias=True)
)


In [33]:
learning_rate = 0.001
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_2.parameters(), lr=learning_rate)

num_epochs = 10
for epoch in range(num_epochs):
    model_2.train()
    total_loss = 0.0
    total_images = 0

    for data, targets in train_loader:
        data = data.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()
        scores = model_2(data)
        loss = criterion(scores, targets)
        loss.backward()
        optimizer.step()

        batch_size = data.size(0)
        total_loss += loss.item() * batch_size
        total_images += batch_size

    average_loss = total_loss / total_images
    print( f"Epoch: {epoch+1} | Train loss: {average_loss:.6f}")

Epoch: 1 | Train loss: 0.303478
Epoch: 2 | Train loss: 0.089584
Epoch: 3 | Train loss: 0.063722
Epoch: 4 | Train loss: 0.053933
Epoch: 5 | Train loss: 0.044481
Epoch: 6 | Train loss: 0.038897
Epoch: 7 | Train loss: 0.034683
Epoch: 8 | Train loss: 0.030234
Epoch: 9 | Train loss: 0.027488
Epoch: 10 | Train loss: 0.023584


In [ ]:
model.eval()

correct = 0
total = 0
total_loss = 0.0

with torch.no_grad(): 
    for data, targets in val_loader:
        data = data.to(device)
        targets = targets.to(device)

        scores = model_2(data)         
        loss = criterion(scores, targets)

        predictions = scores.argmax(dim=1) 

        correct += (predictions == targets).sum().item()
        total += targets.size(0)
        total_loss += loss.item() * targets.size(0)

accuracy = 100 * correct / total
average_loss = total_loss / total


print(f"Validation accuracy: {accuracy:.3f}%")
print(f"Validation loss:     {average_loss:.7f}")

Validation accuracy: 98.408%
Validation loss:     0.0624283
